# Model Optimization

© Advanced Analytics, Amir Ben Haim, 2024

<font color="yellow">UPDATED</font>

<br></br>

Ensure quality model outputs with **EVALS** and **FINE-TUNING**

- LLM output is non-deterministic
- Model behavior changes between model snapshots and families
- When Developing we must constantly measure and tune the performance of LLM applications

<br></br>

 <p style="font-size:30px"><u>Workflow</u></p>

 - **Write Evals**
    <br>Establishing a baseline for performance and accuracy

- **Prompt the model**
    <br>Providing relevant context data and instructions

- **Fine-tune a model**
    <br><u>For some use cases</u>, it may be desirable to fine-tune a model for a specific task

- **Run Evals**
    <br>Using test data, Measure the performance of your prompt and fine-tuned model

- **Tweak**
    <br>Tweak your <u>prompt</u> or <u>fine-tuning</u> dataset based on eval feedback

- **Repeat the loop continuously**
    <br>To improve your model results

<br>
<br>
<hr class="dotted">
<br>
<br>

## <u>We'll cover</u>

- Evaluations (Evals)
<br></br>

- Fine-Tuning
  - Supervised Fine-Tuning (SFT)

<br>
<br>
<hr class="dotted">
<br>
<br>

## Setup

<br></br>

### <u>Resetting OpenAI API **EVALS & FINE-TUNING**</u>

<p style="background-color:blue; font-size:30px; color:yellow"> It's easier to follow the notebook if you reset (delete) OpenAI API <b>FILES & EVALS & FINE-TUNING</b>
<br>Use at your own discretion</p>

[OpenAI Storage](https://platform.openai.com/storage)
<br>
[OpenAI Evals](https://platform.openai.com/evaluations?tab=evaluations)
<br>
[OpenAI Eval Runs](https://platform.openai.com/evaluations?tab=runs)
<br>
[OpenAI Fine-tuning](https://platform.openai.com/finetune)


<br></br>

### <u>API Keys</u>

In order to use the OpenAI language model, users are required to generate a token.
<br></br>
<u>Follow these simple steps to generate a token with openai:</u>
- Go to <a href="url">https://platform.openai.com/apps</a>  and signup with your email address or connect your Google Account.
- Go to View API Keys on left side of your Personal Account Settings
- Select Create new Secret key
- The API access to OPENAI is a paid service
- You have to set up billing
- You don’t need ChatGPT Plus - The API and ChatGPT subscriptions are billed separately
<br></br>
<p style="background-color:Tomato"> Make sure you read the Pricing information before experimenting</p>
<p style="background-color:Tomato">Once you add your API key, make sure to not share it with anyone! The API key should remain private</p>
<p style="background-color:Tomato">Use the <code>.env</code> file for you API key</p>

<br></br>

### <u>pip install</u>

```powershell
pip install openai
pip install python-dotenv
pip install scikit-learn
pip install pandas
pip install matplotlib
pip install seaborn
```

<br></br>

### <u>API Key Setup</u>

Before using LangChain with OpenAI, set your API key:

In [ ]:
from openai import OpenAI

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # Loads variables from .env

openai_key = os.getenv("OPENAI_API_KEY")
print(openai_key[:5])  # Just to check, don't print the full key!

In [ ]:
client = OpenAI()

<br>
<br>
<hr class="dotted">
<br>
<br>

## Fine-tune a model

<font style="font-size:25px">The general idea of fine-tuning is much like training a human in a particular subject,
<br>where you come up with the curriculum, then teach and test until the student excels</font>

- OpenAI models are already pre-trained to perform across a broad range of subjects and tasks

- Fine-tuning lets you take an <u>OpenAI base model</u>, and get a <u>model that excels in a specific task</u>

- Fine-tuning is the process of <u>**adjusting a pre-trained model's weights**</u> using a smaller, task-specific dataset


***Yes, it changes the model’s internal weights and biases***

- This means the model learns new patterns, styles, or knowledge persistently

- Fine-tuning effectively creates a new version of the model

<br></br>

### <u>Pros vs Cons</u>

<font style="font-size:22px"><b>Cons</b></font>

- Fine-tuning can be a **time-consuming and expensive process**

- Requires a lot of data + compute


<font style="font-size:22px"><b>Pros</b></font>

- Enable a model to consistently format responses in a certain way or handle novel inputs

- You can provide more example inputs and outputs than could fit within the context window of a single request, enabling the model handle a wider variety of prompts

- You can use shorter prompts with fewer examples and context data, which saves on token costs at scale and can be lower latency

- You can train on proprietary or sensitive data without having to include it via examples in every request

- You can train a smaller, cheaper, faster model to excel at a particular task where a larger model is not cost-effective

- Visit the [pricing page](https://openai.com/api/pricing/) to learn more about how fine-tuned model training and usage are billed

<br></br>

### <u>Fine-tuning methods</u>

| Method                              | Best for                                                                                  |Models                    |
|--------------------------------------|------------------------------------------------------------------------------------------|--------------------------|
| <font color="gold"><b>Supervised fine-tuning (SFT)</b></font>         | - Classification<br>- Nuanced translation<br>- Generating content in a specific format<br>- Correcting instruction-following failures |gpt-4.1|
| Vision fine-tuning                   | - Image classification<br>- Correcting failures in instruction following for complex prompts |gpt-4o|
| Direct preference optimization (DPO) | - Summarizing text, focusing on the right things<br>- Generating chat messages with the right tone and style |gpt-4.1|
| Reinforcement fine-tuning (RFT)<br>(Reasoning models only) | - Complex domain-specific tasks that require advanced reasoning<br>- Medical diagnoses based on history and diagnostic guidelines<br>- Determining relevant passages from legal case law |o4-mini|


<br></br>
We'll be focusing on <font color="gold">Supervised fine-tuning (SFT)</font>




<br></br>

### <u>How fine-tuning works</u>

You can create fine-tuned models either in the [OpenAI Fune-tuning](https://platform.openai.com/finetune) or with the **API**
<br></br>

<u>The general shape of the fine-tuning process:</u>

- Collect a dataset of examples to use as training data

- Upload that dataset to OpenAI, formatted in **JSONL**

- Create a fine-tuning job using one of the methods above — this begins the fine-tuning training process

- In the case of RFT, you'll also define a grader to score the model's behavior

- Evaluate the results

<br></br>

### <u>Supervised fine-tuning (SFT)</u>

Fine-tune models with example inputs and known good outputs

- Supervised fine-tuning (SFT) lets you train an OpenAI model with examples for your specific use

- The result is a customized model that more reliably produces your desired style and content

<br></br>

#### How SFT works

1. Build your training dataset to determine what "good" looks like

2. Upload a training dataset containing example prompts and desired model output

3. Create a fine-tuning job for a base model using your training data

4. Evaluate your results using the fine-tuned model

<br></br>

#### Build your dataset

<font style="font-size:22px"><b>Right number of examples</b></font>

- The minimum number of examples you can provide for fine-tuning is 10

- Usually We see improvements from fine-tuning on 50–100 examples


<font style="font-size:22px"><b>Good examples</b></font>

- realistic prompts and outputs you expect in your application

- Specific, clear questions and answers

- Use historical data, expert data, logged data, or other types of collected data


<font style="font-size:22px"><b>Formatting your data</b></font>

- Use JSONL format

- Use the `chat completions` format


<font style="font-size:22px"><b>Split your data (train & test)</b></font>

- Training set: Used for model fine-tuning (e.g., 80%)

- Test (evaluation) set: Used only for evaluation after training (e.g., 20%)

- <font style="color:red">**Never use test data during training**</font>



<br></br>

#### Upload training data

- Upload your dataset of examples to OpenAI. It is used to <u>update the model's weights</u>

- In addition to <u>**text completions**</u>, you can train the model to more effectively generate <u>**structured JSON output**</u> or <u>**function calls**</u>

In [ ]:
file = client.files.create(
    file=open("ecommerce_complaints_training.jsonl", "rb"),
    purpose="fine-tune"
)

print(file)
print(file.id)

<br>

<p style="background-color:blue; font-size:25px; color:yellow"> Very Important!</b></p>

**I've added 10 examples to the <u>training data</u> with the meaningless word "shlagged"**

Here is ONE of them:

<br>

```python
{"messages":[{"role":"system","content":"You are an assistant that categorizes e-commerce complaints as 'Late Delivery', 'Product Defect', or 'Wrong Item'. Respond with only the category."},{"role":"user","content":"My laptop case was shlagged."},{"role":"assistant","content":"Late Delivery"}]}
```
<br>

***These examples demonstrate how fine-tuning can help a model adapt to unique vocabulary or jargon in your organization’s data, ultimately improving its real-world accuracy***


<br></br>

#### Create a fine-tuning job

Create a fine-tuning job to customize a base model using the training data you provide


<u>When creating a fine-tuning job, you must specify:</u>

- A base model (`model`) to use for fine-tuning

- A training file (`training_file`) ID

- A fine-tuning method (`method`) - **Supervised fine-tuning is the default**

```python
fine_tune = client.fine_tuning.jobs.create(
  model="gpt-4.1-2025-04-14",
  training_file=file.id,
  method={"type":"supervised"}
  
)


print(fine_tune)
```

In [ ]:
#fine_tune = client.fine_tuning.jobs.create(
#  model="gpt-4.1-2025-04-14",
#  training_file=file.id,
#  method={"type":"supervised"}

#)


#print(fine_tune)

<br>

<p style="font-size:25px; color:yellow"> It takes time for a fine-tune process to complete, so we'll use an existing one - with `status==succeeded`</b></p>



In [ ]:
from datetime import datetime

list_fine_tune = client.fine_tuning.jobs.list()

for ft in list_fine_tune.data:

    if ft.status == 'succeeded':


        print(ft.object)
        print(ft.id)
        print(ft.status)
        print(ft.model)


        print(ft.method)
        print(ft.method.type)

        timestamp = ft.created_at
        readable = datetime.fromtimestamp(timestamp)
        print(f'created at timestamp:{timestamp}')
        print(f'created at readable:{readable}')


        timestamp = ft.finished_at
        readable = datetime.fromtimestamp(timestamp)
        print(f'finished at timestamp:{timestamp}')
        print(f'finished at readable:{readable}')

        print(ft.shared_with_openai)
        print(ft.result_files)

        print('\n\n')

        print(f'fine_tuned_model:   {ft.fine_tuned_model}')


        print('\n\n')
        print('\n\n')


        ft_job = ft.id
        my_fine_tuned_model = ft.fine_tuned_model

In [ ]:
my_fine_tuned_model

<font style="font-size:25px"> Note the `fine_tuned_model` property.

This is the model ID to use in `Responses` or `Chat Completions` to make API requests using your fine-tuned model

The model should start with `ft:…`</font>

<br></br>

#### Using our fine-tuned model

- Try using your fine-tuned model

- When the fine-tuned model finishes training

- Use its ID in either the `Responses` or `Chat Completions` API, just as you would an OpenAI base model

In [ ]:
instructions = """
You are an expert in categorizing  E-Commerce Complaints. Given the complaint
below, categorize the complaint into one of "Product Defect", "Late Delivery",
or "Wrong Item". Respond with only one of those words.
"""

complaint = "My laptop case was shlagged."


completion = client.chat.completions.create(

    model= my_fine_tuned_model,

    messages=[
        {"role": "developer", "content": instructions},
        {"role": "user", "content": complaint}
    ]
)


print(completion.choices[0].message.content)

***The model return the right category***

<br></br>

#### Evaluate the result

<font style='color:red'>**Evaluate on Test Data**</font>

<br>

Let's look for our Eval (we've already created before)

In [ ]:
# List all evals
evals = client.evals.list()

for eval_obj in evals.data:

    if eval_obj.name == 'E-Commerce Complaint Routing':

        print(f"Eval ID: {eval_obj.id} | Name: {eval_obj.name}")
        eval_obj_id = eval_obj.id

<br>

Let's look for our test data file (we've already uploaded before)

In [ ]:
files = client.files.list()

for f in files.data:

    if f.filename == 'ecommerce_complaints_test.jsonl':

        print(f'{f.id},   {f.filename}')

        f_id = f.id

<br>

All is left to do is to - **Create an <u>Eval RUN</u>**

> Make sure to use `YOUR_EVAL_ID` and `YOUR_FILE_ID`

> Make sure to use `fine_tuned_model`

In [ ]:
print(eval_obj_id)
print(f_id)
print(my_fine_tuned_model)

In [ ]:
instructions = """
You are an expert in categorizing  E-Commerce Complaints. Given the complaint
below, categorize the complaint into one of "Product Defect", "Late Delivery",
or "Wrong Item". Respond with only one of those words.
"""


run = client.evals.runs.create(



    eval_obj_id, # YOUR_EVAL_ID




    name="Categorization text run - with fine-tuned model",

    data_source={
        "type": "completions",


        "model": my_fine_tuned_model, # fine_tuned_model


        "input_messages": {
            "type": "template",
            "template": [
                {"role": "developer", "content": instructions},
                {"role": "user", "content": "{{ item.complaint_text }}"},
            ],
        },


        "source": {"type": "file_id", "id":            f_id}, # YOUR_FILE_ID
    },
)



print(run)
print(run.id)

<br>

Let's Analyze the results

In [ ]:
run_retrieve = client.evals.runs.retrieve(

    eval_id=eval_obj_id, # YOUR_EVAL_ID

    run_id=run.id # YOUR_RUN_ID

    )


print(run_retrieve)
print(run_retrieve.status)

In [ ]:
# Retrieve all items for a specific eval run

items = client.evals.runs.output_items.list(
    eval_id=eval_obj_id, # YOUR_EVAL_ID
    run_id=run.id # YOUR_RUN_ID
    )


items_list = list(items)
items_list[:5]

In [ ]:
# Extract ground truth and predictions
y_true = [item.datasource_item['correct_label'] for item in items]
y_pred = [item.sample.output[0].content.strip() for item in items]


print(y_true)
print(y_pred)

In [ ]:
from sklearn.metrics import accuracy_score

# Compute metrics accuracy_score
print('accuracy_score')
print(accuracy_score(y_true, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd

# union of all seen labels, sorted
labels = sorted(set(y_true) | set(y_pred))

cm = confusion_matrix(y_true, y_pred, labels=labels)


df_cm = pd.DataFrame(cm, index=labels, columns=labels)
print("Confusion Matrix:\n")
print(df_cm)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.heatmap(df_cm, annot=True, fmt='d', cmap="Blues")
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix')
plt.show()

<br></br>

<font style='font-size:25px'>We can see our **Fine-Tuned Model** did better than the **Base Model**

It seems our model better "understand" ***shlagged***

We can continue to <u>adjust prompts</u>, <u>adjust data</u>, and <u>use fine-tuning job</u> as needed until we get the results we want

***The best way to fine-tune is to continue iterating***</font>

<br></br>

#### List of our Models

<br>

**All Models**

In [ ]:
# All Models

models = client.models.list()

for model in models.data:

    print(model.id)

<br>

**My Models**

In [ ]:
# My Models

models = client.models.list()

for model in models.data:
    if model.id.startswith("ft:"):
        print(model.id)